# DSA4213 Assignment 1 — Part III data preparation: Rotten Tomatoes sentiment

Builds clean, leakage-checked **train / valid / test** splits for the downstream Fresh-vs-Rotten classifier.

| Step | What it does | Why |
|---|---|---|
| 1. Clean | drop empty reviews, strip `[Full review in Spanish]` tags, drop reviews under `MIN_TOKENS` tokens | the shortest reviews ("Why?", "d") carry almost no signal |
| 2. Deduplicate | drop repeated review texts (normalised), and drop every copy if the copies carry **different** labels | stops the same review from appearing in both train and test |
| 3. WikiText overlap filter | remove reviews whose word n-grams overlap the **WikiText-2 train split** that the language models were trained on | some Wikipedia film articles quote critic reviews word for word |
| 4. Split by movie | assign whole movies to train / valid / test (80 / 10 / 10), then cap reviews per movie and balance the labels | no film has reviews in more than one split |
| 5. Checks + save | asserts that splits share no movies and no texts; writes CSVs and a statistics file for the report | |

**Requires**
* `data/raw/rotten_tomatoes_critic_reviews.csv` from Kaggle: [stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset](https://www.kaggle.com/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset). Sign in, download, and unzip it into `data/raw/`. It is not redistributed in this repo.
* `data/processed/train.txt` (WikiText-2 train, from Part I or `python data/prepare_data.py`).

Writes `data/prepared/rt_{train,valid,test}.csv`, which Part III reads, plus `prep_stats.json` and `overlap_removed.csv`. Runs on CPU; no GPU needed.

Uses the **same tokenizer** as Parts I and II. Fully deterministic given `SEED`.

In [ ]:
# ==========================================
# CELL 0: SETUP (Drive + paths) — same as Part II
# ==========================================
import os, sys, re, json, random
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/dsa4213/ind_assignment")
    
else:
    PROJECT_DIR = Path.cwd()
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

In [16]:
# ==========================================
# CELL 1: CONFIG
# ==========================================
SOURCE = "critic_reviews"   # "critic_reviews": one row per review (recommended)
                            # "movies":         critics_consensus text, one row per movie

DATA_DIR  = Path("data/raw")         # unzipped Kaggle CSVs
WIKI_DIR  = Path("data/processed")   # WikiText-2 splits from Part I / data/prepare_data.py
OUT_DIR   = Path("data/prepared"); OUT_DIR.mkdir(parents=True, exist_ok=True)   # read by Part III
RAW_FILE  = {"critic_reviews": DATA_DIR / "rotten_tomatoes_critic_reviews.csv",
             "movies":         DATA_DIR / "rotten_tomatoes_movies.csv"}[SOURCE]

SEED          = 42
SPLIT_FRAC    = (0.8, 0.1, 0.1)    # fraction of MOVIES in train / valid / test
TARGET_SIZE   = {"train": 20000, "valid": 2500, "test": 2500}   # reviews after balancing (upper bound)
MAX_PER_MOVIE = 10                 # cap so franchises (e.g. 50+ U.N.C.L.E. reviews) don't dominate
MIN_TOKENS    = 4                  # drop ultra-short reviews

# Overlap filter. A review is flagged if EITHER
#   (a) at least OVERLAP_FRAC of its N_GRAM-grams occur in WikiText-2 train, or
#   (b) it shares any LONG_NGRAM-gram with WikiText-2 train (a long verbatim quote).
N_GRAM, OVERLAP_FRAC, LONG_NGRAM = 8, 0.5, 13

In [17]:
# ==========================================
# CELL 2: SHARED TOKENIZER + VOCAB (identical to Parts I–II)
# ==========================================
START_TOKEN, END_TOKEN, PAD_TOKEN, UNK_TOKEN = "<s>", "</s>", "<pad>", "<unk>"
TOKEN_PATTERN = re.compile(r"\w+(?:'\w+)?|[^\w\s]")
WIKITEXT_MARKERS = re.compile(r" @([-,.])@ ")

def clean_wikitext(line):
    return WIKITEXT_MARKERS.sub(r"\1", line)

def tokenize(text):
    return TOKEN_PATTERN.findall(clean_wikitext(text).lower().replace("’", "'"))

wiki_lines = [t for t in (tokenize(l) for l in (WIKI_DIR / "train.txt").read_text(encoding="utf-8").splitlines()) if t]
wiki_counts = Counter(t for line in wiki_lines for t in line)
lm_vocab = set(t for t, _ in wiki_counts.most_common(30000))   # same 30k types as the LMs
print(f"WikiText-2 train: {len(wiki_lines):,} lines, {sum(map(len, wiki_lines)):,} tokens")

WikiText-2 train: 17,556 lines, 2,031,623 tokens


In [18]:
# ==========================================
# CELL 3: LOAD + CLEAN
# ==========================================
stats = {"source": SOURCE, "raw_file": RAW_FILE.name, "seed": SEED}

if SOURCE == "critic_reviews":
    raw = pd.read_csv(RAW_FILE, usecols=["rotten_tomatoes_link", "review_type", "review_content"])
    df = raw.rename(columns={"rotten_tomatoes_link": "movie_id", "review_content": "text"})
    df["label"] = df["review_type"].map({"Fresh": 1, "Rotten": 0})
else:
    raw = pd.read_csv(RAW_FILE, usecols=["rotten_tomatoes_link", "critics_consensus", "tomatometer_status"])
    df = raw.rename(columns={"rotten_tomatoes_link": "movie_id", "critics_consensus": "text"})
    df["label"] = df["tomatometer_status"].map({"Fresh": 1, "Certified-Fresh": 1, "Rotten": 0})
stats["raw_rows"] = len(df)

df = df.dropna(subset=["movie_id", "text", "label"]).copy()
df["label"] = df["label"].astype(int)
# translated reviews carry a tag that is not part of the review
df["text"] = (df["text"].str.replace(r"\[\s*full review in [^\]]*\]", "", flags=re.I, regex=True)
                        .str.replace(r"\s+", " ", regex=True).str.strip())
df["tokens"] = df["text"].map(tokenize)
df["n_tokens"] = df["tokens"].map(len)
stats["after_dropna"] = len(df)
df = df[df["n_tokens"] >= MIN_TOKENS].reset_index(drop=True)
stats["after_min_tokens"] = len(df)
print(json.dumps(stats, indent=1))

{
 "source": "critic_reviews",
 "raw_file": "rotten_tomatoes_critic_reviews.csv",
 "seed": 42,
 "raw_rows": 1130017,
 "after_dropna": 1064211,
 "after_min_tokens": 1058330
}


In [19]:
# ==========================================
# CELL 4: DEDUPLICATION
# ==========================================
# Normalised key: lowercase, letters/digits only, single spaces.
# "Enough, already." and "Enough already" therefore count as the same review.
def norm_key(text):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", text.lower().replace("’", "'"))).strip()

df["key"] = df["text"].map(norm_key)
grp = df.groupby("key")
n_copies = grp["label"].transform("size")
n_labels = grp["label"].transform("nunique")

conflict = n_labels > 1                         # same text, different labels -> unreliable, drop all copies
stats["dup_groups"] = int((grp.size() > 1).sum())
stats["dup_rows_removed"] = int(((n_copies > 1) & ~conflict).sum() - df.loc[(n_copies > 1) & ~conflict, "key"].nunique())
stats["conflicting_label_rows_removed"] = int(conflict.sum())

print("Most common duplicated texts:")
display(df[n_copies > 1].groupby("key").agg(copies=("text", "size"), movies=("movie_id", "nunique"),
                                             labels=("label", lambda s: sorted(set(s))), example=("text", "first"))
          .sort_values("copies", ascending=False).head(10))

df = df[~conflict].drop_duplicates("key", keep="first").reset_index(drop=True)
stats["after_dedup"] = len(df)
print({k: stats[k] for k in ("dup_groups", "dup_rows_removed", "conflicting_label_rows_removed", "after_dedup")})

Most common duplicated texts:


,copies,movies,labels,example
key,,,,
full review at movies for the masses,201,182,"[0, 1]",full review at Movies for the Masses
full review in greek,104,92,"[0, 1]",full review in Greek
click for full review,45,39,"[0, 1]",click for full review
click to read review,43,38,"[0, 1]",Click to read review
click to read the article,28,28,"[0, 1]",Click to read the article.
one of the best films of the year,24,22,[1],One of the best films of the year.
no quote available,22,18,"[0, 1]",No quote available.
a pleasant surprise,19,19,[1],A pleasant surprise.
click to read the full review,19,15,"[0, 1]",click to read the full review


{'dup_groups': 112438, 'dup_rows_removed': 113200, 'conflicting_label_rows_removed': 700, 'after_dedup': 944430}


In [20]:
# ==========================================
# CELL 5: OVERLAP FILTER AGAINST WIKITEXT-2 TRAIN
# ==========================================
def ngram_set(lines, n):
    return {hash(tuple(line[i:i + n])) for line in lines for i in range(len(line) - n + 1)}

wiki_n    = ngram_set(wiki_lines, N_GRAM)
wiki_long = ngram_set(wiki_lines, LONG_NGRAM)
print(f"WikiText-2 train: {len(wiki_n):,} distinct {N_GRAM}-grams, {len(wiki_long):,} distinct {LONG_NGRAM}-grams")

def overlap(tokens):
    grams = [hash(tuple(tokens[i:i + N_GRAM])) for i in range(len(tokens) - N_GRAM + 1)]
    frac = sum(g in wiki_n for g in grams) / len(grams) if grams else 0.0   # shorter than N_GRAM -> not checkable
    long_hit = any(hash(tuple(tokens[i:i + LONG_NGRAM])) in wiki_long for i in range(len(tokens) - LONG_NGRAM + 1))
    return frac, long_hit

ov = df["tokens"].map(overlap)
df["overlap_frac"] = ov.map(lambda x: x[0])
df["long_ngram_hit"] = ov.map(lambda x: x[1])
df["flagged"] = (df["overlap_frac"] >= OVERLAP_FRAC) | df["long_ngram_hit"]

stats["overlap_rule"] = f">= {OVERLAP_FRAC:.0%} of {N_GRAM}-grams in WikiText-2 train, or any {LONG_NGRAM}-gram match"
stats["overlap_any_ngram_hit"] = int((df["overlap_frac"] > 0).sum())   # for context: mostly titles/stock phrases
stats["overlap_flagged_removed"] = int(df["flagged"].sum())
flagged = df[df["flagged"]].sort_values("overlap_frac", ascending=False)
flagged[["movie_id", "label", "overlap_frac", "long_ngram_hit", "text"]].to_csv(OUT_DIR / "overlap_removed.csv", index=False)
print(f"Reviews with ANY {N_GRAM}-gram hit: {stats['overlap_any_ngram_hit']:,} | flagged & removed: {stats['overlap_flagged_removed']:,}")
display(flagged[["movie_id", "overlap_frac", "long_ngram_hit", "text"]].head(15))

df = df[~df["flagged"]].reset_index(drop=True)
stats["after_overlap_filter"] = len(df)

WikiText-2 train: 1,896,442 distinct 8-grams, 1,825,154 distinct 13-grams
Reviews with ANY 8-gram hit: 497 | flagged & removed: 64


,movie_id,overlap_frac,long_ngram_hit,text
344970,m/gi_joe_retaliation,1.000000,True,The first film you could at least laugh at. Th...
572940,m/not_quite_hollywood,0.875000,True,An incredibly energetic and merrily messed-up ...
854507,m/there_will_be_blood,0.750000,False,One of the most wholly original American movie...
813562,m/the_man_from_uncle,0.615385,True,"""The Man From U.N.C.L. E."" is a dull affair."
813561,m/the_man_from_uncle,0.571429,True,"""The Man From U.N.C.L. E."" is totally enjoyabl..."
159619,m/beowulf,0.538462,False,This isn't your great-great-great-great-great-...
813766,m/the_man_from_uncle,0.500000,True,The Man from U.N.C.L.E. fulfills its mission t...
813656,m/the_man_from_uncle,0.461538,True,Against all odds The Man From U.N.C.L.E. is su...
813817,m/the_man_from_uncle,0.400000,True,THE MAN FROM U.N.C.L.E. Has The Style But Refu...
572939,m/not_quite_hollywood,0.380952,True,I'm not sure how this will appeal to a general...


In [21]:
# ==========================================
# CELL 6: SPLIT BY MOVIE, CAP PER MOVIE, BALANCE LABELS
# ==========================================
rng = np.random.default_rng(SEED)
movies = np.array(sorted(df["movie_id"].unique()))
rng.shuffle(movies)
n_tr = int(SPLIT_FRAC[0] * len(movies)); n_va = int(SPLIT_FRAC[1] * len(movies))
movie_split = {m: "train" for m in movies[:n_tr]}
movie_split.update({m: "valid" for m in movies[n_tr:n_tr + n_va]})
movie_split.update({m: "test" for m in movies[n_tr + n_va:]})
df["split"] = df["movie_id"].map(movie_split)

def build_split(part, target, seed):
    # 1) at most MAX_PER_MOVIE reviews per movie (random), 2) equal Fresh/Rotten up to target
    part = part.sample(frac=1, random_state=seed).groupby("movie_id", sort=False).head(MAX_PER_MOVIE)
    per_class = min(target // 2, part["label"].value_counts().min())
    out = pd.concat([part[part.label == c].sample(per_class, random_state=seed) for c in (0, 1)])
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)

splits = {s: build_split(df[df.split == s], TARGET_SIZE[s], SEED + i) for i, s in enumerate(("train", "valid", "test"))}

In [22]:
# ==========================================
# CELL 7: LEAKAGE CHECKS + SAVE
# ==========================================
names = list(splits)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        assert not set(splits[a].movie_id) & set(splits[b].movie_id), f"movie overlap {a}/{b}"
        assert not set(splits[a].key) & set(splits[b].key), f"text overlap {a}/{b}"
print("OK: no movie and no (normalised) review text appears in more than one split")

cols = ["movie_id", "label", "n_tokens", "text"]
for s, d in splits.items():
    d[cols].to_csv(OUT_DIR / f"rt_{s}.csv", index=False)
    toks = [t for ts in d["tokens"] for t in ts]
    stats[f"{s}_reviews"] = len(d)
    stats[f"{s}_movies"] = int(d["movie_id"].nunique())
    stats[f"{s}_fresh_share"] = round(float(d["label"].mean()), 3)
    stats[f"{s}_median_tokens"] = int(d["n_tokens"].median())
    stats[f"{s}_unk_rate"] = round(sum(t not in lm_vocab for t in toks) / len(toks), 4)  # domain shift vs WikiText vocab

(OUT_DIR / "prep_stats.json").write_text(json.dumps(stats, indent=1))
summary = pd.DataFrame({s: {k.split("_", 1)[1]: v for k, v in stats.items() if k.startswith(s + "_")} for s in names}).astype(object)
print(json.dumps({k: v for k, v in stats.items() if not k.startswith(tuple(names))}, indent=1))
summary

OK: no movie and no (normalised) review text appears in more than one split
{
 "source": "critic_reviews",
 "raw_file": "rotten_tomatoes_critic_reviews.csv",
 "seed": 42,
 "raw_rows": 1130017,
 "after_dropna": 1064211,
 "after_min_tokens": 1058330,
 "dup_groups": 112438,
 "dup_rows_removed": 113200,
 "conflicting_label_rows_removed": 700,
 "after_dedup": 944430,
 "overlap_rule": ">= 50% of 8-grams in WikiText-2 train, or any 13-gram match",
 "overlap_any_ngram_hit": 497,
 "overlap_flagged_removed": 64,
 "after_overlap_filter": 944366
}


,train,valid,test
reviews,20000.0,2500.0,2500.0
movies,10831.0,1347.0,1342.0
fresh_share,0.5,0.5,0.5
median_tokens,24.0,24.0,23.0
unk_rate,0.0852,0.0849,0.0861
